**Part 1**

Testing class on data. First, import my .py file to define my class.

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
import pyspark.sql.types as T
import pandas as pd

In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config('spark.sql.ansi.enabled', 'false').getOrCreate()

In [3]:
import Project2_Part1_WarrenE

In [47]:
import importlib
importlib.reload(Project2_Part1_WarrenE)

<module 'Project2_Part1_WarrenE' from '/home/jupyter-eawarre7@ncsu.edu/Project2_Part1_WarrenE.py'>

In [48]:
csv_data = 'air.csv'
my_instance = Project2_Part1_WarrenE. \
        SparkDataCheck.from_csv(spark, \
        "ID integer, Date string, Time timestamp, \
        CO integer, S1 integer, NMHC integer, \
        C6H6 integer, S2 integer, NOX integer, S3 integer, \
        NO2 integer, S4 integer, S5 integer, T double, \
        RH double, AH double", csv_data)
my_instance.df = my_instance.df \
        .withColumnsRenamed({"": "ID", \
                             "CO(GT)": "CO", \
                             "PT08.S1(CO)": "S1", \
                             "NMHC(GT)": "NMHC", \
                             "C6H6(GT)": "C6H6", \
                             "PT08.S2(NMHC)": "S2", \
                             "NOx(GT)": "NOX", \
                             "PT08.S3(NOx)": "S3", \
                             "NO2(GT)": "NO2", \
                             "PT08.S4(NO2)": "S4", \
                             "PT08.S5(O3)": "S5"}) \
        .withColumn("over100", \
        F.when(my_instance.df["NMHC"] > 100, "over 100")
        .otherwise("less/equal to 100")) \
        .replace(-200, None)

In [43]:
my_instance.df.columns

['ID',
 'Date',
 'Time',
 'CO',
 'S1',
 'NMHC',
 'C6H6',
 'S2',
 'NOX',
 'S3',
 'NO2',
 'S4',
 'S5',
 'T',
 'RH',
 'AH',
 'over100']

In [42]:
my_instance.df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO: integer (nullable = true)
 |-- S1: integer (nullable = true)
 |-- NMHC: integer (nullable = true)
 |-- C6H6: integer (nullable = true)
 |-- S2: integer (nullable = true)
 |-- NOX: integer (nullable = true)
 |-- S3: integer (nullable = true)
 |-- NO2: integer (nullable = true)
 |-- S4: integer (nullable = true)
 |-- S5: integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- over100: string (nullable = false)



In [49]:
my_instance.check_limits("CO", None, None).df.show(3)

No upper or lower bounds provided
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+
only showing top 3 rows


26/03/29 08:27:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [45]:
my_instance.check_limits("CO", None, 10).df.show(3)

+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|inBounds|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|    NULL|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|    true|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    NULL|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
only showing top 3 rows


26/03/29 08:24:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [46]:
my_instance.check_limits("CO", 1.5, None).df.show(3)

+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|inBounds|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|    NULL|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|    true|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    NULL|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
only showing top 3 rows


26/03/29 08:25:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [28]:
my_instance.check_limits("CO", 1.5, 4).df.show(3)

+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|inBounds|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|    NULL|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|    true|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    NULL|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
only showing top 3 rows


26/03/29 08:13:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [29]:
my_instance.check_string("over100", ["over 100"]).df.show(3)

+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|inBounds|inLevels|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|    NULL|    true|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|    true|    true|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    NULL|   false|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


26/03/29 08:13:48 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [30]:
my_instance.check_string("over100", ["over 100", "less/equal to 100"]).df.show(3)

+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
| ID|     Date|               Time|  CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|inBounds|inLevels|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|  0|3/10/2004|2026-03-29 18:00:00|NULL|1360| 150|NULL|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|    NULL|    true|
|  1|3/10/2004|2026-03-29 19:00:00|   2|1292| 112|NULL| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|    true|    true|
|  2|3/10/2004|2026-03-29 20:00:00|NULL|1402|  88|   9| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    NULL|    true|
+---+---------+-------------------+----+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


26/03/29 08:13:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: ID, Date, Time, CO, S1, NMHC, C6H6, S2, NOX, S3, NO2, S4, S5, T, RH, AH
Expected: ID but found: 
CSV file: file:///home/jupyter-eawarre7@ncsu.edu/air.csv


In [204]:
my_instance.check_string("CO(GT)", ["over 100"]).df.show(3)

Supplied column is not string type. No changes have been made to the dataframe.
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3

In [205]:
my_instance.check_string("CO(GT)", None).df.show(3)

Supplied column is not string type. No changes have been made to the dataframe.
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3

In [206]:
my_instance.check_missing("CO(GT)").df.show(3)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|isMissing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|    false|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         over 100|    true|

In [207]:
my_instance.check_missing("T").df.show(3)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|isMissing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|    false|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         over 100|    true|

In [208]:
my_instance.check_missing("RH").df.show(3)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|isMissing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|    false|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         over 100|    true|

In [209]:
my_instance.check_missing("Date").df.show(3)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
| ID|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|          over100|inBounds|inLevels|isMissing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------------+--------+--------+---------+
|  0|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         over 100|    true|    true|    false|
|  1|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         over 100|    true|

In [210]:
my_instance.get_minmax("RH")

,min(RH),max(RH)
0,9.2,88.7


In [211]:
my_instance.get_minmax("RH", "over100")

,over100,min(RH),max(RH)
0,less/equal to 100,9.2,88.7
1,over 100,14.9,81.8


In [212]:
my_instance.get_minmax("Date")

Error: The supplied column is not numeric type


In [11]:
my_instance.get_minmax(None)

{"ts": "2026-03-28 22:31:27.968", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `PT08`.`S1(CO)` cannot be resolved. Did you mean one of the following? [`PT08.S1(CO)`, `PT08.S3(NOx)`, `PT08.S4(NO2)`, `PT08.S5(O3)`, `PT08.S2(NMHC)`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o86.agg.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `PT08`.`S1(CO)` cannot be resolved. Did you mean one of the following? [`PT08.S1(CO)`, `PT08.S3(NOx)`, `PT08.S4(NO2)`, `PT08.S5(O3)`, `PT08.S2(NMHC)`]. SQLSTATE: 42703;\n'Aggregate [unresolvedalias('min('PT08.S1(CO))), unresolvedalias('

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `PT08`.`S1(CO)` cannot be resolved. Did you mean one of the following? [`PT08.S1(CO)`, `PT08.S3(NOx)`, `PT08.S4(NO2)`, `PT08.S5(O3)`, `PT08.S2(NMHC)`]. SQLSTATE: 42703;
'Aggregate [unresolvedalias('min('PT08.S1(CO))), unresolvedalias('max('PT08.S1(CO)))]
+- Project [CASE WHEN (cast(ID#34 as double) = -200.0) THEN cast(null as int) ELSE ID#34 END AS ID#36, Date#18, Time#19, CASE WHEN (CO(GT)#20 = -200.0) THEN cast(null as double) ELSE CO(GT)#20 END AS CO(GT)#37, CASE WHEN (cast(PT08.S1(CO)#21 as double) = -200.0) THEN cast(null as int) ELSE PT08.S1(CO)#21 END AS PT08.S1(CO)#38, CASE WHEN (cast(NMHC(GT)#22 as double) = -200.0) THEN cast(null as int) ELSE NMHC(GT)#22 END AS NMHC(GT)#39, CASE WHEN (C6H6(GT)#23 = -200.0) THEN cast(null as double) ELSE C6H6(GT)#23 END AS C6H6(GT)#40, CASE WHEN (cast(PT08.S2(NMHC)#24 as double) = -200.0) THEN cast(null as int) ELSE PT08.S2(NMHC)#24 END A